# Vectors

In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

V = torch.tensor([[2.0, 1.0], [-1.0, 2.0], [1.5, -1.5]])

fig, ax = plt.subplots(figsize=(4.8, 4.8))

ax.axhline(0, color="grey", lw=1)
ax.axvline(0, color="grey", lw=1)

for v in V:
    ax.quiver(0, 0, v[0], v[1], angles="xy", scale_units="xy", scale=1, width=0.007)

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect("equal")
plt.show()

## Norms

The length of the arrow is its norm.

In [ ]:
for v in V:
    l1 = float(v.abs().sum())
    l2 = float(v.norm())
    print(f"{v.tolist()}  L1 {l1:.3f}   L2 {l2:.3f}")

In [ ]:
t = torch.linspace(0, 2 * torch.pi, 400)
circle = torch.stack([torch.cos(t), torch.sin(t)], 1)
diamond = circle / circle.abs().sum(1, keepdim=True)

fig, ax = plt.subplots(figsize=(4.8, 4.8))

ax.plot(circle[:, 0], circle[:, 1], lw=2, label="L2 = 1")
ax.plot(diamond[:, 0], diamond[:, 1], lw=2, label="L1 = 1")

ax.axhline(0, color="grey", lw=1)
ax.axvline(0, color="grey", lw=1)
ax.set_aspect("equal")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=2, frameon=False)
plt.show()

In [ ]:
from matplotlib.lines import Line2D

angles = torch.tensor([0, 30, 60, 90, 135, 250], dtype=torch.float32) * torch.pi / 180
V = torch.stack([torch.cos(angles), torch.sin(angles)], 1)

V2 = V / V.norm(dim=1, keepdim=True)
V1 = V / V.abs().sum(1, keepdim=True)

fig, ax = plt.subplots(figsize=(5.8, 5.8))

ax.plot(circle[:, 0], circle[:, 1], lw=1.5, color="C0", alpha=0.45)
ax.plot(diamond[:, 0], diamond[:, 1], lw=1.5, color="C1", alpha=0.45)

box = dict(facecolor="white", edgecolor="none", alpha=0.75, pad=1.0)

for i in range(len(V)):
    ax.quiver(0, 0, *V2[i], angles="xy", scale_units="xy", scale=1,
              width=0.007, color="C0", headwidth=4, headlength=5)
    ax.quiver(0, 0, *V1[i], angles="xy", scale_units="xy", scale=1,
              width=0.007, color="C1", headwidth=4, headlength=5)

    u = V2[i] / V2[i].norm()
    p = torch.tensor([-u[1], u[0]])

    ax.text(*(V2[i] + u * 0.17), f"L1 {float(V2[i].abs().sum()):.2f}",
            color="C0", fontsize=7.5, ha="center", va="center", bbox=box)
    ax.text(*(V1[i] + p * 0.22), f"L2 {float(V1[i].norm()):.2f}",
            color="C1", fontsize=7.5, ha="center", va="center", bbox=box)

ax.plot(V2[:, 0], V2[:, 1], "o", ms=5, mfc="none", mec="C0", mew=1.5, zorder=6)
ax.plot(V1[:, 0], V1[:, 1], "o", ms=5, color="C1", zorder=5)

ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlim(-1.45, 1.45)
ax.set_ylim(-1.45, 1.45)
ax.set_aspect("equal")
ax.legend(handles=[Line2D([], [], color="C0", lw=2), Line2D([], [], color="C1", lw=2)],
          labels=["normalised to L2 = 1", "normalised to L1 = 1"],
          loc="upper center", bbox_to_anchor=(0.5, -0.03), ncol=2, frameon=False)
plt.show()

## Dot product and angle

The dot product measures how much two vectors point the same way. Normalised, it becomes
the cosine of the angle between them.

In [ ]:
a = torch.tensor([2.0, 1.0])
b = torch.tensor([1.0, 2.0])

dot = float(a @ b)
cos = dot / float(a.norm() * b.norm())
angle = torch.acos(torch.tensor(cos)) * 180 / torch.pi

print("a . b  =", dot)
print("cos    =", cos)
print("angle  =", float(angle), "degrees")

### Projection

Projecting `a` onto `b` keeps the part of `a` that lies along `b`. The leftover part is
perpendicular to `b`.

In [ ]:
proj = (a @ b) / (b @ b) * b
resid = a - proj

fig, ax = plt.subplots(figsize=(5.4, 4.9))

ax.quiver(0, 0, *b, angles="xy", scale_units="xy", scale=1,
          width=0.006, color="grey", alpha=0.6, label="b")
ax.text(float(b[0]) + 0.10, float(b[1]) + 0.06, "b", color="grey", fontsize=11)

ax.quiver(0, 0, *proj, angles="xy", scale_units="xy", scale=1,
          width=0.009, color="C2", label="proj  (along b)")
ax.quiver(*proj, *resid, angles="xy", scale_units="xy", scale=1,
          width=0.009, color="C1", label="resid  (perpendicular to b)")
ax.quiver(0, 0, *a, angles="xy", scale_units="xy", scale=1,
          width=0.009, color="C0", label="a")

u = b / b.norm()
v = resid / resid.norm()
corner = proj + (u + v) * 0.10
ax.plot([(proj + u * 0.10)[0], corner[0], (proj + v * 0.10)[0]],
        [(proj + u * 0.10)[1], corner[1], (proj + v * 0.10)[1]], color="grey", lw=1)

ax.text(0.03, 0.97,
        f"a = ({a[0]:.0f}, {a[1]:.0f})    b = ({b[0]:.0f}, {b[1]:.0f})\n"
        f"a · b = {float(a @ b):.0f}      b · b = {float(b @ b):.0f}\n"
        f"a · b / b · b = {float(a @ b / (b @ b)):.1f}\n"
        f"proj = {float(a @ b / (b @ b)):.1f} · b = ({proj[0]:.1f}, {proj[1]:.1f})",
        transform=ax.transAxes, va="top", fontsize=9,
        bbox=dict(facecolor="white", edgecolor="grey", alpha=0.92))

ax.set_xlim(-0.4, 2.5)
ax.set_ylim(-0.4, 2.3)
ax.set_aspect("equal")
ax.set_title("b points the axis; a splits into proj + resid")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

## Cosine similarity

Length is thrown away and only direction is kept. Two vectors pointing the same way score
1.

In [ ]:
pairs = [
    ("same direction, different length", [1.0, 2.0], [2.0, 4.0]),
    ("perpendicular",                    [1.0, 0.0], [0.0, 1.0]),
    ("opposite",                         [1.0, 1.0], [-1.0, -1.0]),
]

for label, u, w in pairs:
    u = torch.tensor(u)
    w = torch.tensor(w)
    print(f"{label:32} {float((u @ w) / (u.norm() * w.norm())):+.3f}")

In [ ]:
from matplotlib.patches import Arc

u = torch.tensor([1.0, 0.5])
examples = [
    ("same direction", torch.tensor([2.0, 1.0])),
    ("perpendicular", torch.tensor([-0.5, 1.0])),
    ("obtuse", torch.tensor([-1.0, 0.5])),
]

fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.7))

for ax, (title, v) in zip(axes, examples):
    cos = float((u @ v) / (u.norm() * v.norm()))
    a1 = float(torch.atan2(u[1], u[0]) * 180 / torch.pi)
    a2 = float(torch.atan2(v[1], v[0]) * 180 / torch.pi)

    ax.quiver(0, 0, *v, angles="xy", scale_units="xy", scale=1, width=0.018, color="C1")
    ax.quiver(0, 0, *u, angles="xy", scale_units="xy", scale=1, width=0.018, color="C0")

    lo, hi = min(a1, a2), max(a1, a2)
    if hi - lo > 1:
        ax.add_patch(Arc((0, 0), 1.1, 1.1, angle=0, theta1=lo, theta2=hi,
                         color="grey", lw=1.4))

    ax.set_title(title)
    ax.text(0.2, -1.30, f"cos = {cos:+.2f}", ha="center", fontsize=13)

    ax.set_xlim(-1.6, 2.4)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect("equal")
    ax.axhline(0, color="grey", lw=0.6)
    ax.axvline(0, color="grey", lw=0.6)

plt.tight_layout()
plt.show()